# Experimental fluxomics nullspace check

In [1]:
# Automatic module reloading
%load_ext autoreload
%autoreload 2

# Packages
import os
import sys
import pandas as pd 
import cobra
import numpy as np

# Directories
ROOT_DIR = os.path.abspath('..')

if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)
    
experimental_data_dir = os.path.join(ROOT_DIR, "data", "experimental", "fluxes_W3110_MD121_M9+Glu_none.csv")

## 1. Using all experimental reactions

In [2]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)
experimental_data = pd.read_csv(experimental_data_dir)

# Convert DataFrame to dictionary where rxn_id is key and exp_flux is value
experimental_flux_dict = dict(zip(experimental_data['rxn_id'], experimental_data['exp_flux']))
print(f"Dictionary length: {len(experimental_flux_dict)}")

Dictionary length: 143


**Biomass fix**

The fluxomics study used the wild-type strain E. coli K-12 MG1655, so the biomass growth rate value will only be set to this reaction: BIOMASS_Ec_iML1515_WT_75p37M

In [3]:
# Delete the core reaction from the dictionary
del experimental_flux_dict['BIOMASS_Ec_iML1515_core_75p37M']
print(f"Dictionary length: {len(experimental_flux_dict)}")

Dictionary length: 142


**Filter transport reactions**

The mapped fluxomics data currently assigns the flux value of exchange reactions (EX_) to transport reactions ('tpp', 'tex', 'rpp'). This should be discarded when using experimental data to constrain a model, because the measured flux corresponds to just the exchange reaction. 

DOUBT: Is it valid to keep for transport reactions when doing a GEM vs experimental flux comparison?

In [4]:
transport_suffixes = ('tpp', 'tex', 'rpp')

print(f"Dict before removing transport reactions: {len(experimental_flux_dict)}")

experimental_flux_dict = {
    rxn_id: data for rxn_id, data in experimental_flux_dict.items()
    if not rxn_id.endswith(transport_suffixes)
}

print(f"Dict after removing transport reactions: {len(experimental_flux_dict)}")

Dict before removing transport reactions: 142
Dict after removing transport reactions: 132


**Apply experimental fluxes as bounds**

TO DO: Get the LB/UB from the Crown SI Excel ([1,2]Glc column)
    * Will have to ask Krishna how to treat these bounds. Some of them 
    are reported as (0,Inf).

In [5]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              stdev_factor=0.02,
                                              verbose=True)

c:\Users\lyachinas\miniconda3\envs\kingems\Lib\site-packages\bioservices\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-11-18 11:56:23.238 | INFO     | kinGEMs.config:<module>:12 - PROJ_ROOT path is: C:\Users\lyachinas\OneDrive - University of Toronto\GitHub\kinGEMs_v2


SHK3Dr: Kept experimental lower bound 0.1274 (orig: -1000.0)
SHK3Dr: Kept experimental upper bound 0.1326 (orig: 1000.0)

G5SD: Kept experimental lower bound 0.1568 (orig: 0.0)
G5SD: Kept experimental upper bound 0.1632 (orig: 1000.0)

CS: Kept experimental lower bound 2.0873999999999997 (orig: 0.0)
CS: Kept experimental upper bound 2.1726 (orig: 1000.0)

ICDHyr: Kept experimental lower bound 1.9796 (orig: -1000.0)
ICDHyr: Kept experimental upper bound 2.0604 (orig: 1000.0)

PPCK: Kept experimental lower bound 0.0294 (orig: 0.0)
PPCK: Kept experimental upper bound 0.0306 (orig: 1000.0)

ME1: Kept experimental lower bound 0.0294 (orig: 0.0)
ME1: Kept experimental upper bound 0.0306 (orig: 1000.0)

ALATA_L: Kept experimental lower bound -0.3774 (orig: -1000.0)
ALATA_L: Kept experimental upper bound -0.3626 (orig: 1000.0)

ASPTA: Kept experimental lower bound -1.6218000000000001 (orig: -1000.0)
ASPTA: Kept experimental upper bound -1.5582 (orig: 1000.0)

PYK: Kept experimental lower bound

In [6]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [7]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
13dpg_c
mocogdp_c
frulysp_c
progly_c
clpn161_p
citr__L_c
thmpp_c
murein5p5p_p
adocbl_c
xu5p__D_c
aspsa_c
fadh2_c
gp4g_c
cgly_c
3pg_c
ctp_c
xu5p__L_c
murein4px4p4p_p
phpyr_c
argsuc_c
chor_c
hom__L_c
4hbz_c
pe160_p
adpglc_c
clpn160_p
clpn181_p
ru5p__D_c
man6pglyc_c
colipa_e
pe181_p
3ig3p_c
dctp_c
utp_c
ahcys_c
uaagmda_c
3mob_c
hemeO_c
gmhep7p_c
2dmmq8_c
gtspmd_c
uppg3_c
mococdp_c
mlthf_c
2ippm_c
3psme_c
murein5p5p5p_p
fmn_c
q8_c
pphn_c
ap5a_c
pheme_c
so4_c
so4_p
pe161_p
rephaccoa_c
2fe2s_c
btn_c
malthp_c
nadph_c
4fe4s_c
icit_c
no3_c
dgtp_c
lipoamp_c
enter_c
seramp_c
5fthf_c


In [8]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Metabolite: 13dpg_c
Constrained reactions involving this metabolite:
  GAPD: coeff=+1.00, flux=17.1300, net=+17.1300
    g3p_c + nad_c + pi_c --> 13dpg_c + h_c + nadh_c
  PGK: coeff=+1.00, flux=-0.8100, net=-0.8100
    3pg_c + atp_c <-- 13dpg_c + adp_c

Metabolite: mocogdp_c
Constrained reactions involving this metabolite:
  BIOMASS_Ec_iML1515_WT_75p37M: coeff=-0.00, flux=0.7500, net=-0.0000
    0.000223 10fthf_c + 0.000223 2dmmql8_c + 2.5e-05 2fe2s_c + 0.000248 4fe4s_c + 0.000223 5mthf_c + 0.000279 accoa_c + 0.000223 adocbl_c + 0.499149 ala__L_c + 0.000223 amet_c + 0.28742 arg__L_c + 0.234232 asn__L_c + 0.234232 asp__L_c + 75.55223 atp_c + 2e-06 btn_c + 0.004952 ca2_c + 0.000223 chor_c + 0.004952 cl_c + 0.002944 clpn160_p + 0.00229 clpn161_p + 0.00118 clpn181_p + 0.000168 coa_c + 2.4e-05 cobalt2_c + 0.008151 colipa_e + 0.129799 ctp_c + 0.000674 cu2_c + 0.088988 cys__L_c + 0.024805 datp_c + 0.025612 dctp_c + 0.025612 dgtp_c + 0.024805 dttp_c + 0.000223 enter_c + 0.000223 fad_c + 0.006

## 2. Using JUST medium reactions (exchanges)

In [9]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [10]:
medium_flux_dict = {
    "EX_glc__D_e": -10,
    "EX_so4_e": -0.175297,
    "EX_o2_e": -14.2804,
    "EX_nh4_e": -5.23859,
}

medium_bounds = {
    "EX_glc__D_e": (-10.02, -9.98),
    "EX_so4_e": (-0.19, -0.16),
    "EX_o2_e": (-14.98, -13.62),
    "EX_nh4_e": (-5.62, -4.84),
    #"EX_h2o_e": (-6.96, 6.96)
}

In [11]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              medium_flux_dict, 
                                              experimental_bounds=medium_bounds)

EX_glc__D_e: Kept original lower bound -10.0 (exp: -10.02)
EX_glc__D_e: Kept experimental upper bound -9.98 (orig: 1000.0)

EX_so4_e: Kept experimental lower bound -0.19 (orig: -1000.0)
EX_so4_e: Kept experimental upper bound -0.16 (orig: 1000.0)

EX_o2_e: Kept experimental lower bound -14.98 (orig: -1000.0)
EX_o2_e: Kept experimental upper bound -13.62 (orig: 1000.0)

EX_nh4_e: Kept experimental lower bound -5.62 (orig: -1000.0)
EX_nh4_e: Kept experimental upper bound -4.84 (orig: 1000.0)



In [12]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model, fluxes_to_print=medium_flux_dict)

SUCCESS: The experimental data is CONSISTENT with the model.
             fluxes
EX_glc__D_e  -10.00
EX_so4_e      -0.16
EX_o2_e      -14.98
EX_nh4_e      -5.62


# Takeaways

- The experimental mapping needs correction
    - Back when I did these mappings I assigned to multiple iML1515 reactions the same flux value of a single measured experimental reaction - because this single reaction represented a cluster of multiple reactions in iML1515.
    - To address this:
        1. Go back to the interim dataframes in the fluxomics mapping (BactoCat project) and check which cluster reactions I expanded against `problematic_reactions` - remove these
        2. Re-run pipeline. There should be no problem (hopefully)
            - If there are still problematic  `problematic_reactions`: how to address these?
    - These corrections are necessary for a flux-to-flux comparison on many datapoints. A way around this would be to keep just the very few reactions that aren't lumped (about 20).

- All media bounds are properly mapped
    - Solution is feasible


# Remapping experimental fluxes

Remapped the experimental fluxes to their GEM counterpart. 

This was done using NotebookLM, a csv with the iML1515 reactions, a csv with the experimental reactions, and prompting 'map the entries in  'EXP REACTIONS' using exp_rxn_id, to its counterpart in 'GEM REACTIONS''. 

Furthermore, the data used for fluxomics corresponds to the 'COMPLETE-MFA' column in the Crown study SI Excel. This combines all 14 data sets (each corresponding to an isotopic tracer) to produce a single, unified flux map. This approach resolved more fluxes with smaller confidence intervals than any single tracer could.

The csv with the mapping and COMPLETE-MFA fluxes (with CIs) is under /data/experimental/crown_fluxomics_mapped.csv

The final csv to use is under /data/experimental/crown_fluxomics_final.csv. This was:
- cleaned to keep a single exp_rxn mapped to its gem_rxn
- all fluxes were divided by a factor of 10 (to match GEM 10 mmol glucose uptake)

In [13]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [14]:
final_experimental_data_dir = os.path.join(ROOT_DIR, "data", "experimental", "crown_fluxomics_final.csv")
experimental_data = pd.read_csv(final_experimental_data_dir)

# Convert DataFrame to dictionary where rxn_id is key and exp_flux is value
experimental_flux_dict = dict(zip(experimental_data['rxn_id'], experimental_data['exp_flux']))
print(f"Dictionary length: {len(experimental_flux_dict)}")

Dictionary length: 46


In [15]:
bounds_tuples = zip(
    experimental_data['exp_flux_lb'],
    experimental_data['exp_flux_ub'], 
)

# Zip the reaction IDs with the flux tuples
experimental_bounds_dict = dict(zip(experimental_data['rxn_id'], bounds_tuples))

In [16]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              verbose=True, 
                                              experimental_bounds=experimental_bounds_dict)

GLCptspp: Kept experimental lower bound 9.98049 (orig: 0.0)
GLCptspp: Kept experimental upper bound 10.0195 (orig: 1000.0)

PGI: Kept experimental lower bound 7.05523 (orig: -1000.0)
PGI: Kept experimental upper bound 7.26009 (orig: 1000.0)

PFK: Kept experimental lower bound 8.20648 (orig: 0.0)
PFK: Kept experimental upper bound 8.32198 (orig: 1000.0)

FBA: Kept experimental lower bound 8.20648 (orig: -1000.0)
FBA: Kept experimental upper bound 8.32198 (orig: 1000.0)

TPI: Kept experimental lower bound 8.20648 (orig: -1000.0)
TPI: Kept experimental upper bound 8.32198 (orig: 1000.0)

PYK: Kept experimental lower bound 2.28389 (orig: 0.0)
PYK: Kept experimental upper bound 3.14417 (orig: 1000.0)

G6PDH2r: Kept experimental lower bound 2.58343 (orig: -1000.0)
G6PDH2r: Kept experimental upper bound 2.79442 (orig: 1000.0)

GND: Kept experimental lower bound 2.44507 (orig: 0.0)
GND: Kept experimental upper bound 2.65122 (orig: 1000.0)

RPE: Kept experimental lower bound 1.06981 (orig: -100

In [17]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [18]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
xu5p__D_c
dimp_c
imp_c
hom__L_c
4r5au_c
din_c
ahcys_c
murein5p5p5p_p
ap5a_c
2ddara_c
25aics_c


In [19]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Metabolite: xu5p__D_c
Constrained reactions involving this metabolite:
  RPE: coeff=+1.00, flux=1.1524, net=+1.1524
    ru5p__D_c --> xu5p__D_c
  TKT2: coeff=-1.00, flux=-0.4404, net=+0.4404
    e4p_c + xu5p__D_c <-- f6p_c + g3p_c
  TKT1: coeff=-1.00, flux=-0.7120, net=+0.7120
    r5p_c + xu5p__D_c <-- g3p_c + s7p_c

Reactions not in experimental data with this metabolite:
  XYLK: coeff=+1.00
    atp_c + xylu__D_c --> adp_c + h_c + xu5p__D_c
  RBP4E: coeff=+1.00
    ru5p__L_c <=> xu5p__D_c

Reactions not in experimental data with this metabolite:
  NTPP10: coeff=+1.00
    ditp_c + h2o_c --> dimp_c + h_c + ppi_c
  NTD12: coeff=-1.00
    dimp_c + h2o_c --> din_c + pi_c

Reactions not in experimental data with this metabolite:
  NTPP9: coeff=+1.00
    h2o_c + itp_c --> h_c + imp_c + ppi_c
  IMPC: coeff=-1.00
    h2o_c + imp_c <=> fprica_c
  NTD11: coeff=-1.00
    h2o_c + imp_c --> ins_c + pi_c
  IMPD: coeff=-1.00
    h2o_c + imp_c + nad_c --> h_c + nadh_c + xmp_c
  HXPRT: coeff=+1.00
   

## Removing problematic reactions

From the print above: TKT2, TKT1, RPE, SUCDi, ASNS2

In [20]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [21]:
final_experimental_data_dir = os.path.join(ROOT_DIR, "data", "experimental", "crown_fluxomics_final.csv")
experimental_data = pd.read_csv(final_experimental_data_dir)

# Convert DataFrame to dictionary where rxn_id is key and exp_flux is value
experimental_flux_dict = dict(zip(experimental_data['rxn_id'], experimental_data['exp_flux']))
print(f"Dictionary length: {len(experimental_flux_dict)}")


Dictionary length: 46


In [22]:
# keys_to_remove = ['TKT1', 'TKT2', 'RPE', 'ASNS2', 'THRD']
reactions_to_remove = ['GP4GH', 'AP5AH', 'FBA3', 'PFK_3']

print("Removing problematic reactions:")
for rxn_id in reactions_to_remove:
    try:
        rxn = model.reactions.get_by_id(rxn_id)
        print(f"  {rxn_id}: {rxn.reaction}")
        model.remove_reactions([rxn])
    except KeyError:
        print(f"  {rxn_id}: not found in model")

print(f"\nModel size: {len(model.reactions)} → {len(model.reactions)} reactions")


Removing problematic reactions:
  GP4GH: gp4g_c + h2o_c --> 2.0 gdp_c + 2.0 h_c
  AP5AH: ap5a_c + h2o_c --> adp_c + atp_c + 2.0 h_c
  FBA3: s17bp_c <=> dhap_c + e4p_c
  PFK_3: atp_c + s7p_c --> adp_c + h_c + s17bp_c

Model size: 2708 → 2708 reactions


In [23]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              verbose=True, 
                                              experimental_bounds=experimental_bounds_dict)

GLCptspp: Kept experimental lower bound 9.98049 (orig: 0.0)
GLCptspp: Kept experimental upper bound 10.0195 (orig: 1000.0)

PGI: Kept experimental lower bound 7.05523 (orig: -1000.0)
PGI: Kept experimental upper bound 7.26009 (orig: 1000.0)

PFK: Kept experimental lower bound 8.20648 (orig: 0.0)
PFK: Kept experimental upper bound 8.32198 (orig: 1000.0)

FBA: Kept experimental lower bound 8.20648 (orig: -1000.0)
FBA: Kept experimental upper bound 8.32198 (orig: 1000.0)

TPI: Kept experimental lower bound 8.20648 (orig: -1000.0)
TPI: Kept experimental upper bound 8.32198 (orig: 1000.0)

PYK: Kept experimental lower bound 2.28389 (orig: 0.0)
PYK: Kept experimental upper bound 3.14417 (orig: 1000.0)

G6PDH2r: Kept experimental lower bound 2.58343 (orig: -1000.0)
G6PDH2r: Kept experimental upper bound 2.79442 (orig: 1000.0)

GND: Kept experimental lower bound 2.44507 (orig: 0.0)
GND: Kept experimental upper bound 2.65122 (orig: 1000.0)

RPE: Kept experimental lower bound 1.06981 (orig: -100

In [24]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [25]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
xu5p__D_c
dimp_c
imp_c
hom__L_c
4r5au_c
din_c
ahcys_c
murein5p5p5p_p
2ddara_c
25aics_c
ap4a_c


In [26]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Metabolite: xu5p__D_c
Constrained reactions involving this metabolite:
  TKT2: coeff=-1.00, flux=-0.4404, net=+0.4404
    e4p_c + xu5p__D_c <-- f6p_c + g3p_c
  TKT1: coeff=-1.00, flux=-0.7120, net=+0.7120
    r5p_c + xu5p__D_c <-- g3p_c + s7p_c
  RPE: coeff=+1.00, flux=1.1524, net=+1.1524
    ru5p__D_c --> xu5p__D_c

Reactions not in experimental data with this metabolite:
  RBP4E: coeff=+1.00
    ru5p__L_c <=> xu5p__D_c
  XYLK: coeff=+1.00
    atp_c + xylu__D_c --> adp_c + h_c + xu5p__D_c

Reactions not in experimental data with this metabolite:
  NTPP10: coeff=+1.00
    ditp_c + h2o_c --> dimp_c + h_c + ppi_c
  NTD12: coeff=-1.00
    dimp_c + h2o_c --> din_c + pi_c

Reactions not in experimental data with this metabolite:
  ADSS: coeff=-1.00
    asp__L_c + gtp_c + imp_c --> dcamp_c + gdp_c + 2.0 h_c + pi_c
  IMPD: coeff=-1.00
    h2o_c + imp_c + nad_c --> h_c + nadh_c + xmp_c
  GMPR: coeff=+1.00
    gmp_c + 2.0 h_c + nadph_c --> imp_c + nadp_c + nh4_c
  INSK: coeff=+1.00
    atp_c +

## Keeping just reactions from ECOMICS

In [27]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [28]:
crown_file = os.path.join(ROOT_DIR, "data", "experimental", "crown_fluxomics_final.csv")

ecomics_file = os.path.join(ROOT_DIR, "data", "experimental", "ecomics_rxns.csv")

In [29]:
df_ecomics = pd.read_csv(ecomics_file)
df_crown = pd.read_csv(crown_file)

# Get unique 'bigg_id' values from ecomics
# .dropna() is added to remove any missing (NaN) values
ecomics_ids = set(df_ecomics['bigg_id'].dropna().unique())

# Get unique 'rxn_id' values from crown
crown_ids = set(df_crown['rxn_id'].dropna().unique())

print(f"Total unique 'bigg_id's found in ecomics: {len(ecomics_ids)}")
print(f"Total unique 'rxn_id's found in crown: {len(crown_ids)}")

Total unique 'bigg_id's found in ecomics: 27
Total unique 'rxn_id's found in crown: 46


In [30]:
# Find all IDs that are in ecomics_ids AND ALSO in crown_ids
common_ids_set = ecomics_ids.intersection(crown_ids)

# Convert back to a list for convenience (printing or iterating)
final_common_ids = list(common_ids_set)

print(f"Found {len(final_common_ids)} common reaction IDs that exist in both files.")

# Optional: Print the first 10 common IDs as a sample
print(f"Common IDs: {final_common_ids[:]}")

Found 21 common reaction IDs that exist in both files.
Common IDs: ['CS', 'PFK', 'FUM', 'FBA', 'SUCDi', 'TKT2', 'TKT1', 'MDH', 'ME1', 'ICDHyr', 'G6PDH2r', 'ICL', 'PYK', 'TPI', 'GND', 'MALS', 'PGI', 'TALA', 'PPC', 'GLCptspp', 'RPI']


In [31]:
mask = df_crown['rxn_id'].isin(common_ids_set)

df_crown_filtered = df_crown[mask]

In [32]:
experimental_flux_dict = dict(zip(df_crown_filtered['rxn_id'], df_crown_filtered['exp_flux']))
print(f"Dictionary length: {len(experimental_flux_dict)}")

bounds_tuples = zip(
    df_crown_filtered['exp_flux_lb'],
    df_crown_filtered['exp_flux_ub'], 
)

# Zip the reaction IDs with the flux tuples
experimental_bounds_dict = dict(zip(df_crown_filtered['rxn_id'], bounds_tuples))

Dictionary length: 21


In [33]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              verbose=True, 
                                              experimental_bounds=experimental_bounds_dict)

GLCptspp: Kept experimental lower bound 9.98049 (orig: 0.0)
GLCptspp: Kept experimental upper bound 10.0195 (orig: 1000.0)

PGI: Kept experimental lower bound 7.05523 (orig: -1000.0)
PGI: Kept experimental upper bound 7.26009 (orig: 1000.0)

PFK: Kept experimental lower bound 8.20648 (orig: 0.0)
PFK: Kept experimental upper bound 8.32198 (orig: 1000.0)

FBA: Kept experimental lower bound 8.20648 (orig: -1000.0)
FBA: Kept experimental upper bound 8.32198 (orig: 1000.0)

TPI: Kept experimental lower bound 8.20648 (orig: -1000.0)
TPI: Kept experimental upper bound 8.32198 (orig: 1000.0)

PYK: Kept experimental lower bound 2.28389 (orig: 0.0)
PYK: Kept experimental upper bound 3.14417 (orig: 1000.0)

G6PDH2r: Kept experimental lower bound 2.58343 (orig: -1000.0)
G6PDH2r: Kept experimental upper bound 2.79442 (orig: 1000.0)

GND: Kept experimental lower bound 2.44507 (orig: 0.0)
GND: Kept experimental upper bound 2.65122 (orig: 1000.0)

RPI: Kept experimental lower bound 1.34422 (orig: -100

In [34]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [35]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
gp4g_c
s17bp_c
ap5a_c


In [36]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Reactions not in experimental data with this metabolite:
  GP4GH: coeff=-1.00
    gp4g_c + h2o_c --> 2.0 gdp_c + 2.0 h_c

Reactions not in experimental data with this metabolite:
  FBA3: coeff=-1.00
    s17bp_c <=> dhap_c + e4p_c
  PFK_3: coeff=+1.00
    atp_c + s7p_c --> adp_c + h_c + s17bp_c

Reactions not in experimental data with this metabolite:
  AP5AH: coeff=-1.00
    ap5a_c + h2o_c --> adp_c + atp_c + 2.0 h_c


In [37]:
reactions_to_remove = ['GP4GH', 'AP5AH', 'FBA3', 'PFK_3']

print(f"\nModel size: {len(constrained_model.reactions)} reactions")

print("Removing problematic reactions:")
for rxn_id in reactions_to_remove:
    try:
        rxn = constrained_model.reactions.get_by_id(rxn_id)
        print(f"  {rxn_id}: {rxn.reaction}")
        constrained_model.remove_reactions([rxn])
    except KeyError:
        print(f"  {rxn_id}: not found in model")

print(f"\nModel size: {len(constrained_model.reactions)} reactions")


Model size: 2712 reactions
Removing problematic reactions:
  GP4GH: gp4g_c + h2o_c --> 2.0 gdp_c + 2.0 h_c
  AP5AH: ap5a_c + h2o_c --> adp_c + atp_c + 2.0 h_c
  FBA3: s17bp_c <=> dhap_c + e4p_c
  PFK_3: atp_c + s7p_c --> adp_c + h_c + s17bp_c

Model size: 2708 reactions


In [38]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [39]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
5pg35pg_c
2ddara_c
ap4a_c


In [40]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Reactions not in experimental data with this metabolite:
  LDGUNPD: coeff=-1.00
    5pg35pg_c + h2o_c --> 2.0 gmp_c + h_c
  CDGUNPD: coeff=+1.00
    cdigmp_c + h2o_c --> 5pg35pg_c + h_c

Reactions not in experimental data with this metabolite:
  2DDARAA: coeff=-1.00
    2ddara_c <=> gcald_c + pyr_c
  DXYLTD: coeff=+1.00
    dxylnt_c <=> 2ddara_c + h2o_c

Reactions not in experimental data with this metabolite:
  AP4AH: coeff=-1.00
    ap4a_c + h2o_c --> 2.0 adp_c + 2.0 h_c
  AP4AS: coeff=+1.00
    2.0 atp_c + h_c --> ap4a_c + ppi_c


In [41]:
reactions_to_remove = ['LDGUNPD', 'CDGUNPD', 'DXYLTD', '2DDARAA', 'AP4AH', 'AP4AS']

print(f"\nModel size: {len(constrained_model.reactions)} reactions")

print("Removing problematic reactions:")
for rxn_id in reactions_to_remove:
    try:
        rxn = constrained_model.reactions.get_by_id(rxn_id)
        print(f"  {rxn_id}: {rxn.reaction}")
        constrained_model.remove_reactions([rxn])
    except KeyError:
        print(f"  {rxn_id}: not found in model")

print(f"\nModel size: {len(constrained_model.reactions)} reactions")


Model size: 2708 reactions
Removing problematic reactions:
  LDGUNPD: 5pg35pg_c + h2o_c --> 2.0 gmp_c + h_c
  CDGUNPD: cdigmp_c + h2o_c --> 5pg35pg_c + h_c
  DXYLTD: dxylnt_c <=> 2ddara_c + h2o_c
  2DDARAA: 2ddara_c <=> gcald_c + pyr_c
  AP4AH: ap4a_c + h2o_c --> 2.0 adp_c + 2.0 h_c
  AP4AS: 2.0 atp_c + h_c --> ap4a_c + ppi_c

Model size: 2702 reactions


In [42]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [43]:
from kinGEMs.nullspace_utils import find_infeasible_constraints
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)
    
problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)

Metabolites causing inconsistency:
adprib_c
e4p_c

Reactions not in experimental data with this metabolite:
  NADN: coeff=+1.00
    h2o_c + nad_c --> adprib_c + h_c + ncam_c
  ADPRDP: coeff=-1.00
    adprib_c + h2o_c --> amp_c + 2.0 h_c + r5p_c

Metabolite: e4p_c
Constrained reactions involving this metabolite:
  TALA: coeff=+1.00, flux=-0.7120, net=-0.7120
    g3p_c + s7p_c <-- e4p_c + f6p_c
  TKT2: coeff=-1.00, flux=-0.4404, net=+0.4404
    e4p_c + xu5p__D_c <-- f6p_c + g3p_c

Reactions not in experimental data with this metabolite:
  E4PP: coeff=-1.00
    e4p_c + h2o_c --> erthrs_c + pi_c
  E4PD: coeff=-1.00
    e4p_c + h2o_c + nad_c <=> 4per_c + 2.0 h_c + nadh_c
  DDPA: coeff=-1.00
    e4p_c + h2o_c + pep_c --> 2dda7p_c + pi_c


### Takeaways

- Measured fluxes are still not in the nullspace. However, the final print to identify the problematic reactions returns nothing. 
- What this means is that there are not experimental reactions mapped to these metabolites. 

Infeasibility occurs because:
- large genome-scale models include alternative pathways that may not be active under specific growth conditions
- these can create false infeasibilities when you tightly constrain the active pathways

To confirm this with Krishna
TO DO: Remove the problematic reactions in iML1515?

## To dos -  Meeting Nov 11th

- Option 1: Fit into E coli core. It has fewer reactions and this might overcome infeasibility due to alternative pathways 
- Option 2: Build GEM from scratch using their network
    - get genes and gprs from E coli core

## *E coli* core model

In [44]:
model_xml = os.path.join(ROOT_DIR, "models", f"e_coli_core.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [45]:
core_rxn_ids = set([rxn.id for rxn in model.reactions])
print(f"Reactions in core model: {sorted(core_rxn_ids)}")

Reactions in core model: ['ACALD', 'ACALDt', 'ACKr', 'ACONTa', 'ACONTb', 'ACt2r', 'ADK1', 'AKGDH', 'AKGt2r', 'ALCD2x', 'ATPM', 'ATPS4r', 'BIOMASS_Ecoli_core_w_GAM', 'CO2t', 'CS', 'CYTBD', 'D_LACt2', 'ENO', 'ETOHt2r', 'EX_ac_e', 'EX_acald_e', 'EX_akg_e', 'EX_co2_e', 'EX_etoh_e', 'EX_for_e', 'EX_fru_e', 'EX_fum_e', 'EX_glc__D_e', 'EX_gln__L_e', 'EX_glu__L_e', 'EX_h2o_e', 'EX_h_e', 'EX_lac__D_e', 'EX_mal__L_e', 'EX_nh4_e', 'EX_o2_e', 'EX_pi_e', 'EX_pyr_e', 'EX_succ_e', 'FBA', 'FBP', 'FORt', 'FORt2', 'FRD7', 'FRUpts2', 'FUM', 'FUMt2_2', 'G6PDH2r', 'GAPD', 'GLCpts', 'GLNS', 'GLNabc', 'GLUDy', 'GLUN', 'GLUSy', 'GLUt2r', 'GND', 'H2Ot', 'ICDHyr', 'ICL', 'LDH_D', 'MALS', 'MALt2_2', 'MDH', 'ME1', 'ME2', 'NADH16', 'NADTRHD', 'NH4t', 'O2t', 'PDH', 'PFK', 'PFL', 'PGI', 'PGK', 'PGL', 'PGM', 'PIt2r', 'PPC', 'PPCK', 'PPS', 'PTAr', 'PYK', 'PYRt2', 'RPE', 'RPI', 'SUCCt2_2', 'SUCCt3', 'SUCDi', 'SUCOAS', 'TALA', 'THD2', 'TKT1', 'TKT2', 'TPI']


In [46]:
final_experimental_data_dir = os.path.join(ROOT_DIR, "data", "experimental", "crown_fluxomics_final_core.csv")
experimental_data = pd.read_csv(final_experimental_data_dir)

In [47]:
# Check overlap
core_rxn_ids = set([rxn.id for rxn in model.reactions])
exp_rxn_ids = set(experimental_data['rxn_id'].unique())

print(f"Core model reactions: {len(core_rxn_ids)}")
print(f"Experimental reactions: {len(exp_rxn_ids)}")
print(f"Overlap: {len(exp_rxn_ids.intersection(core_rxn_ids))}")

not_in_core = exp_rxn_ids - core_rxn_ids
if not_in_core:
    print(f"\nExperimental reactions NOT in core model ({len(not_in_core)}):")
    print(sorted(not_in_core))

Core model reactions: 95
Experimental reactions: 46
Overlap: 35

Experimental reactions NOT in core model (11):
['ALATA_L', 'ASNS2', 'ASPTA', 'EDA', 'EDD', 'EX_so4_e', 'GHMT2r', 'GLYCL', 'MTHFD', 'MTHFR2', 'THRD']


In [48]:
# Filter to only reactions that exist in the model
experimental_data_filtered = experimental_data[experimental_data['rxn_id'].isin(core_rxn_ids)].copy()

# Create dictionaries from filtered data
experimental_flux_dict = dict(zip(
    experimental_data_filtered['rxn_id'], 
    experimental_data_filtered['exp_flux']
))

experimental_bounds_dict = dict(zip(
    experimental_data_filtered['rxn_id'],
    zip(experimental_data_filtered['exp_flux_lb'], experimental_data_filtered['exp_flux_ub'])
))

# Verify the dicts
print(f"experimental_flux_dict: {len(experimental_flux_dict)} entries")
print(f"experimental_bounds_dict: {len(experimental_bounds_dict)} entries")

experimental_flux_dict: 35 entries
experimental_bounds_dict: 35 entries


In [49]:
# Problems might be coming just from TCA cycle. Relax its rxn bounds

# Relax SUCDi
experimental_bounds_dict['SUCDi'] = (0.85, 1.30)  # Was (1.00, 1.18)

# Relax SUCOAS and ICL 
experimental_bounds_dict['SUCOAS'] = (0.30, 0.70)  # Was (0.36, 0.63)
experimental_bounds_dict['ICL'] = (0.15, 0.35)     # Was (0.17, 0.31)

In [50]:
# Adding a tolerance (relax constraints)
tolerance = 0.05  # 5% tolerance

for rxn_id in experimental_bounds_dict.keys():
    lb, ub = experimental_bounds_dict[rxn_id]
    flux = experimental_flux_dict[rxn_id]
    
    slack = abs(flux) * tolerance
    
    experimental_bounds_dict[rxn_id] = (lb - slack, ub + slack)

In [51]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              verbose=True, 
                                              experimental_bounds=experimental_bounds_dict)

GLCpts: Kept experimental lower bound 9.480490999999999 (orig: 0.0)
GLCpts: Kept experimental upper bound 10.519499000000001 (orig: 1000.0)

PGI: Kept experimental lower bound 6.696998499999999 (orig: -1000.0)
PGI: Kept experimental upper bound 7.6183215 (orig: 1000.0)

PFK: Kept experimental lower bound 7.793297500000001 (orig: 0.0)
PFK: Kept experimental upper bound 8.7351625 (orig: 1000.0)

FBA: Kept experimental lower bound 7.793297500000001 (orig: -1000.0)
FBA: Kept experimental upper bound 8.7351625 (orig: 1000.0)

TPI: Kept experimental lower bound 7.793297500000001 (orig: -1000.0)
TPI: Kept experimental upper bound 8.7351625 (orig: 1000.0)

PYK: Kept experimental lower bound 2.1485225 (orig: 0.0)
PYK: Kept experimental upper bound 3.2795375 (orig: 1000.0)

G6PDH2r: Kept experimental lower bound 2.4493739999999997 (orig: -1000.0)
G6PDH2r: Kept experimental upper bound 2.9284760000000003 (orig: 1000.0)

GND: Kept experimental lower bound 2.3180669999999997 (orig: 0.0)
GND: Kept e

In [52]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [53]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
gln__L_c
nh4_c
r5p_c
ru5p__D_c
succ_c
succoa_c
xu5p__D_c
coa_c
e4p_c
f6p_c
fdp_c
fum_c
g6p_c


In [54]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Metabolite: gln__L_c
Constrained reactions involving this metabolite:
  GLNS: coeff=+1.00, flux=0.5078, net=+0.5078
    atp_c + glu__L_c + nh4_c --> adp_c + gln__L_c + h_c + pi_c
  BIOMASS_Ecoli_core_w_GAM: coeff=-0.26, flux=0.7523, net=-0.1924
    1.496 3pg_c + 3.7478 accoa_c + 59.81 atp_c + 0.361 e4p_c + 0.0709 f6p_c + 0.129 g3p_c + 0.205 g6p_c + 0.2557 gln__L_c + 4.9414 glu__L_c + 59.81 h2o_c + 3.547 nad_c + 13.0279 nadph_c + 1.7867 oaa_c + 0.5191 pep_c + 2.8328 pyr_c + 0.8977 r5p_c --> 59.81 adp_c + 4.1182 akg_c + 3.7478 coa_c + 59.81 h_c + 3.547 nadh_c + 13.0279 nadp_c + 59.81 pi_c

Reactions not in experimental data with this metabolite:
  GLNabc: coeff=+1.00
    atp_c + gln__L_e + h2o_c --> adp_c + gln__L_c + h_c + pi_c
  GLUSy: coeff=-1.00
    akg_c + gln__L_c + h_c + nadph_c --> 2.0 glu__L_c + nadp_c
  GLUN: coeff=-1.00
    gln__L_c + h2o_c --> glu__L_c + nh4_c

Metabolite: nh4_c
Constrained reactions involving this metabolite:
  GLNS: coeff=-1.00, flux=0.5078, net=-0.5078
  

### Takeaways
- E coli core model also turned ouyt infeasible
- New approach: compare LB/UB of MFA against GEM FVA
    - This will go around the issue of infeasibility due to the different networks. 
